In [37]:
import warnings
from langchain._api import LangChainDeprecationWarning

warnings.simplefilter("ignore", category=LangChainDeprecationWarning)

In [38]:
import os
from dotenv import load_dotenv,find_dotenv
_ = load_dotenv(find_dotenv())

groq_api_key =  os.environ["GROQ_API_KEY"]

In [39]:
from langchain_groq import ChatGroq
llmModel = ChatGroq(model = "llama3-70b-8192")

In [40]:
from langchain_community.utilities import SQLDatabase

sqlite_db_path="data/street_tree_db.sqlite"

db = SQLDatabase.from_uri(f"sqlite:///{sqlite_db_path}") #sqlite:/// → Tells it to use SQLite.

Step 1: Translating the question in natural langauge into SQL Query

In [41]:
from langchain.chains import create_sql_query_chain

chain=create_sql_query_chain(llmModel,db)

response = chain.invoke({"question": "How many distinct values are in the 'species_name' column of the 'street_trees' table?"})
response

'Here is the answer:\n\nQuestion: How many distinct values are in the \'qSpecies\' column of the \'street_trees\' table?\nSQLQuery: SELECT COUNT(DISTINCT "qSpecies") FROM street_trees;'

In [42]:
db.run('SELECT COUNT(DISTINCT "qSpecies") FROM street_trees;')

'[(148,)]'

In [43]:
chain.get_prompts()[0].pretty_print()

You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most 5 results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result

# Step 2: Executing the SQL Query

In [44]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
write_query = create_sql_query_chain(llmModel,db)
execute_query = QuerySQLDataBaseTool(db=db)

chain = write_query | execute_query

chain.invoke({"question":"List the species of trees that are present in San Francisco"})

'Error: (sqlite3.OperationalError) near "Here": syntax error\n[SQL: Here\'s the answer:\n\nQuestion: List the species of trees that are present in San Francisco\nSQLQuery: SELECT DISTINCT "qSpecies" FROM street_trees \nLIMIT 5;]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)'

In [45]:
db.run('SELECT DISTINCT "qSpecies" FROM street_trees\nLIMIT 5;')

'[("Arbutus \'Marina\' :: Hybrid Strawberry Tree",), (\'Afrocarpus gracilior :: Fern Pine\',), ("Thuja occidentalis \'Emerald\' :: Emerald Arborvitae",), ("Magnolia grandiflora \'Little Gem\' :: Little Gem Magnolia",), (\'Platanus x hispanica :: Sycamore: London Plane\',)]'

# Now we translate the response into natural language response

In [46]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

answer_prompt = PromptTemplate.from_template(
    """Given the following user question, 
    corresponding SQL query, and SQL result, 
    answer the user question.

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer: """
)

chain = (
    RunnablePassthrough.assign(query=write_query).assign(
        result=itemgetter("query") | execute_query
    )
    | answer_prompt
    | llmModel
    | StrOutputParser()
)

chain.invoke({"question": "List the species of trees that are present in San Francisco"})

'I think there\'s a bit of confusion here!\n\nThe SQL query provided is not actually executed, and instead, an error is thrown due to the syntax error in the SQL query. Specifically, the phrase "Here is the answer:" is not a valid SQL syntax.\n\nTo answer the user\'s question, we need a valid SQL query that retrieves the distinct species of trees present in San Francisco. Assuming the table `street_trees` has a column `qSpecies` and another column indicating the location (e.g., `location`), the correct SQL query would be:\n```sql\nSELECT DISTINCT qSpecies\nFROM street_trees\nWHERE location = \'San Francisco\';\n```\nHowever, since we don\'t have the actual SQL result, we cannot provide a specific list of tree species present in San Francisco.'